# populariteit bepalen van tedtalk videos

# eerst de data ophalen en opschonen

In [91]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
import joblib
from _datetime import datetime
import re
from scipy import stats
import plotly.express as px

df = pd.read_csv("Kaggle_TED_video_metadata_balanced.csv")
# df = pd.read_csv("goodKaggle - Kaggle.csv")
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   title          600 non-null    object
 1   tags           590 non-null    object
 2   views          600 non-null    int64 
 3   likes          600 non-null    int64 
 4   dislikes       600 non-null    int64 
 5   comment_count  600 non-null    int64 
 6   published_at   600 non-null    object
 7   duration       600 non-null    object
 8   category_id    600 non-null    int64 
dtypes: int64(5), object(4)
memory usage: 42.3+ KB


,views,likes,dislikes,comment_count,category_id
count,6.000000e+02,6.000000e+02,600.0,600.000000,600.000000
mean,1.066151e+06,2.263473e+04,0.0,1070.870000,25.030000
std,3.611008e+06,1.000825e+05,0.0,4055.503954,4.812244
min,1.000000e+00,0.000000e+00,0.0,0.000000,1.000000
25%,5.922150e+04,6.242500e+02,0.0,86.000000,22.000000
50%,1.332595e+05,1.846000e+03,0.0,202.000000,27.000000
75%,4.101668e+05,6.382250e+03,0.0,511.750000,28.000000
max,5.562224e+07,1.921445e+06,0.0,77980.000000,29.000000


##### met de column title kan het model niet veel. Daarom maak ik er een column van met de hoeveelheid karakters in de titel.

In [92]:
df["title_length"] = df['title'].apply(len)
df.head(5)

,title,tags,views,likes,dislikes,comment_count,published_at,duration,category_id,title_length
0,Stories from a home for terminally ill childre...,"TED Talk,TED Talks,Children,Community,Death,Fa...",77455,1768,0,49,2017-03-24T15:32:48Z,PT15M19S,29,60
1,Why our screens make us less happy | Adam Alter,"TEDTalk,TEDTalks,Addiction,Computers,Interface...",800326,20579,0,572,2017-08-01T15:29:04Z,PT9M30S,22,47
2,A tribute to nurses | Carolyn Jones,"TEDTalk,TEDTalks,Cancer,Community,Compassion,D...",87635,1877,0,48,2017-05-30T18:17:56Z,PT10M49S,22,35
3,"Asking for help is a strength, not a weakness ...","TEDTalk,TEDTalks,Children,Communication,Commun...",190840,4726,0,187,2017-04-12T15:17:51Z,PT11M56S,22,67
4,Don't feel sorry for refugees -- believe in th...,"TEDTalk,TEDTalks,Children,Global issues,Humani...",98523,2669,0,226,2017-07-25T15:06:25Z,PT14M14S,29,62


##### de title is nutteloos voor machine learning, dus die halen we weg. Ook zijn de dislikes allemaal 0, omdat youtube dit uitgeschakeld heeft. Daarom verwijder ik deze column ook.

In [93]:
df = df.drop(columns=["title", "dislikes"], axis=1)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   tags           590 non-null    object
 1   views          600 non-null    int64 
 2   likes          600 non-null    int64 
 3   comment_count  600 non-null    int64 
 4   published_at   600 non-null    object
 5   duration       600 non-null    object
 6   category_id    600 non-null    int64 
 7   title_length   600 non-null    int64 
dtypes: int64(5), object(3)
memory usage: 37.6+ KB


##### de column "tags" bevat 10 null waarden. Hier ga ik echter niks aan doen, omdat dat natuurlijk voorkomt in de dataset. Er worden in de realiteit video's geopload zonder tags. Verder zijn er geen null waarden.

##### De tag column is nog niet leesbaar voor machine learning. Als ik one-hot encoding zou toepassen, heb ik 591 comlumns. Dit is niet handig omdat ik dan weer andere tags heb voor mijn eigen dataset. Daarom maak ik 1 column met de hoeveelheid tags.

In [94]:
df['num_tags'] = df['tags'].apply(lambda x: len(x.split(',')) if isinstance(x, str) else 0)
df = df.drop("tags", axis=1)
df.head(5)

,views,likes,comment_count,published_at,duration,category_id,title_length,num_tags
0,77455,1768,49,2017-03-24T15:32:48Z,PT15M19S,29,60,18
1,800326,20579,572,2017-08-01T15:29:04Z,PT9M30S,22,47,9
2,87635,1877,48,2017-05-30T18:17:56Z,PT10M49S,22,35,15
3,190840,4726,187,2017-04-12T15:17:51Z,PT11M56S,22,67,12
4,98523,2669,226,2017-07-25T15:06:25Z,PT14M14S,29,62,9


##### De published_at column is niet bruikbaar voor machine learning, daarom pas ik er wat feature engineering op toe. Eerst de dag, maand, jaar en uur scheiden.

In [95]:
df['published_at'] = pd.to_datetime(df['published_at'])

df['year_published'] = df['published_at'].dt.year
df['month_published'] = df['published_at'].dt.month
df['day_published'] = df['published_at'].dt.day
df['hour_published'] = df['published_at'].dt.hour

##### Ook kan ik de dag van de week opslaan, en de tijd sinds upload in een aparte column stoppen. de column published at hebben we niet meer nodig.

In [96]:
df['published_at'] = df['published_at'].dt.tz_localize(None)
df['days_since_published'] = (datetime.now() - df['published_at']).dt.days

df = df.drop("published_at", axis= 1)
df.head(5)

,views,likes,comment_count,duration,category_id,title_length,num_tags,year_published,month_published,day_published,hour_published,days_since_published
0,77455,1768,49,PT15M19S,29,60,18,2017,3,24,15,2770
1,800326,20579,572,PT9M30S,22,47,9,2017,8,1,15,2640
2,87635,1877,48,PT10M49S,22,35,15,2017,5,30,18,2703
3,190840,4726,187,PT11M56S,22,67,12,2017,4,12,15,2752
4,98523,2669,226,PT14M14S,29,62,9,2017,7,25,15,2648


#### met de duration kan ik niet veel in deze format. Daarom ga ik het omzetten naar een leesbaar format

In [97]:
df['duration_seconds'] = df['duration'].apply(lambda x: (int(re.match(r'PT(\d+)M', x).group(1)) * 60 if isinstance(x, str) and re.match(r'PT(\d+)M', x) else 0) + 
                                                            (int(re.match(r'PT(\d+)S', x).group(1)) if isinstance(x, str) and re.match(r'PT(\d+)S', x) else 0))
df = df.drop("duration", axis=1)
df.head(3)

,views,likes,comment_count,category_id,title_length,num_tags,year_published,month_published,day_published,hour_published,days_since_published,duration_seconds
0,77455,1768,49,29,60,18,2017,3,24,15,2770,900
1,800326,20579,572,22,47,9,2017,8,1,15,2640,540
2,87635,1877,48,22,35,15,2017,5,30,18,2703,600


# engagement rate, views over tijd, gemiddelde weergaven van category

In [98]:
df['engagement_rate'] = df['likes'] / df['views']

df['views_over_time'] = df['views'] / df['days_since_published']
df.head()


,views,likes,comment_count,category_id,title_length,num_tags,year_published,month_published,day_published,hour_published,days_since_published,duration_seconds,engagement_rate,views_over_time
0,77455,1768,49,29,60,18,2017,3,24,15,2770,900,0.022826,27.962094
1,800326,20579,572,22,47,9,2017,8,1,15,2640,540,0.025713,303.153788
2,87635,1877,48,22,35,15,2017,5,30,18,2703,600,0.021418,32.421384
3,190840,4726,187,22,67,12,2017,4,12,15,2752,660,0.024764,69.345930
4,98523,2669,226,29,62,9,2017,7,25,15,2648,840,0.027090,37.206571


### omdat ik later een scaler nodig heb met de later bepaalde relevante columns, moet ik die fitten en opslaan (ik krijg een error wanneer ik wil scalen en columns mis)

In [99]:
relevant_scaler = StandardScaler()
relevant_columns = ["views", "likes", "comment_count", "engagement_rate","views_over_time"]
relevant_scaler.fit(df[relevant_columns])

StandardScaler()

# De outliers verwijderen

In [100]:
# scatterplot van views en likes om de outliers te bekijken
fig = px.scatter(df, x='views', y='likes',
                 title='views en likes')
fig.show()

In [101]:
df = df[np.abs(stats.zscore(df["views"])) < 1.5]
df = df[np.abs(stats.zscore(df["likes"])) < 1.5]
# outliers rondom views/likes bekijken
fig = px.scatter(df, x='views', y='likes',
                 title='views en likes')
fig.show()

# Alles schalen

In [102]:
scaler = StandardScaler()
columns_to_scale = ["views_over_time", "engagement_rate", "views", "likes", "comment_count", "title_length", "num_tags", "year_published", "month_published", "hour_published", "days_since_published", "duration_seconds", "day_published"]
df[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])
df.head(5)

,views,likes,comment_count,category_id,title_length,num_tags,year_published,month_published,day_published,hour_published,days_since_published,duration_seconds,engagement_rate,views_over_time
0,-0.421694,-0.341057,-0.483092,29,0.533769,0.708833,1.27354,-1.022146,1.003082,-0.425061,-1.193553,0.294972,1.170156,-0.364173
1,1.087769,2.504900,0.367558,22,-0.401787,-0.525614,1.27354,0.506560,-1.598212,-0.425061,-1.296695,-0.640289,1.549404,2.109688
2,-0.400436,-0.324566,-0.484719,22,-1.265377,0.297351,1.27354,-0.410664,1.681681,0.506407,-1.246711,-0.484412,0.985233,-0.324085
3,-0.184929,0.106466,-0.258638,22,1.037530,-0.114132,1.27354,-0.716405,-0.354115,-0.425061,-1.207835,-0.328535,1.424735,0.007851
4,-0.377700,-0.204742,-0.195205,29,0.677700,-0.525614,1.27354,0.200819,1.116182,-0.425061,-1.290348,0.139096,1.730265,-0.281069


# Dataframe opslaan en schaler opslaan voor hergebruik

In [103]:
df.to_csv("./clean_kaggle_data.csv")

scaler_filename = "scaler.save"
joblib.dump(relevant_scaler, scaler_filename) 

['scaler.save']

In [104]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 542 entries, 0 to 599
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   views                 542 non-null    float64
 1   likes                 542 non-null    float64
 2   comment_count         542 non-null    float64
 3   category_id           542 non-null    int64  
 4   title_length          542 non-null    float64
 5   num_tags              542 non-null    float64
 6   year_published        542 non-null    float64
 7   month_published       542 non-null    float64
 8   day_published         542 non-null    float64
 9   hour_published        542 non-null    float64
 10  days_since_published  542 non-null    float64
 11  duration_seconds      542 non-null    float64
 12  engagement_rate       542 non-null    float64
 13  views_over_time       542 non-null    float64
dtypes: float64(13), int64(1)
memory usage: 63.5 KB


In [105]:
df.head()

,views,likes,comment_count,category_id,title_length,num_tags,year_published,month_published,day_published,hour_published,days_since_published,duration_seconds,engagement_rate,views_over_time
0,-0.421694,-0.341057,-0.483092,29,0.533769,0.708833,1.27354,-1.022146,1.003082,-0.425061,-1.193553,0.294972,1.170156,-0.364173
1,1.087769,2.504900,0.367558,22,-0.401787,-0.525614,1.27354,0.506560,-1.598212,-0.425061,-1.296695,-0.640289,1.549404,2.109688
2,-0.400436,-0.324566,-0.484719,22,-1.265377,0.297351,1.27354,-0.410664,1.681681,0.506407,-1.246711,-0.484412,0.985233,-0.324085
3,-0.184929,0.106466,-0.258638,22,1.037530,-0.114132,1.27354,-0.716405,-0.354115,-0.425061,-1.207835,-0.328535,1.424735,0.007851
4,-0.377700,-0.204742,-0.195205,29,0.677700,-0.525614,1.27354,0.200819,1.116182,-0.425061,-1.290348,0.139096,1.730265,-0.281069
